In [5]:
import importlib
import NHL_script
importlib.reload(NHL_script)
import NHL_data
importlib.reload(NHL_data)

# # NHL SCHEDULE
# NHL_script.make_todays_schedule()

# # NHL YESTERDAYS SCORES FROM NHL_API
# NHL_script.process_yesterdays_scores_to_report()

# # NHL DATA SCRAPE FROM MONEYPUCK
# NHL_data.get_nhl_skaters()
# NHL_data.get_nhl_goalies()
# NHL_data.get_nhl_lines()
# NHL_data.get_nhl_teams()

# Process Data
NHL_data.process_nhl_data_and_generate_html()

Combined data saved to NHL_data/combined_nhl_skaters.csv
Filtered HTML table saved to NHL_data/skaters_html_table.html


In [ ]:
import importlib
import NHL_script
importlib.reload(NHL_script)
import NHL_data
importlib.reload(NHL_data)
import json

teams = [
"ANA",
"BOS",
"BUF",
"CAR",
"CBJ",
"CGY",
"CHI",
"COL",
"DAL",
"DET",
"EDM",
"FLA",
"LAK",
"MIN",
"MTL",
"NJD",
"NSH",
"NYI",
"NYR",
"OTT",
"PHI",
"PIT",
"SEA",
"SJS",
"STL",
"TBL",
"TOR",
"UTA",
"VAN",
"VGK",
"WPG",
"WSH"
]

def get_nhl_team_roster(team):
    """
    Fetch NHL standings (current) via requests and cache the JSON response.

    Uses:
    - requests.get with allow_redirects to match `curl -L -X GET`
    - cache.get_or_fetch to reuse a fresh cached file

    Tune freshness via config.CACHE_EXPIRY_HOURS (in your CacheManager).
    """
    cache_key = f"nhl_{team}_roster"

    def fetch_standings():
        # url = "https://api-web.nhle.com/v1/standings/now"
        url = f"https://api-web.nhle.com/v1/roster/{team}/current"
        resp = requests.get(
            url,
            timeout=30,
            allow_redirects=True,
            headers={"Accept": "application/json"},
        )
        resp.raise_for_status()
        return resp.json()

    return cache.get_or_fetch(cache_key, fetch_standings)

for x in teams:
    get_nhl_team_roster(x)

NameError: name 'cache' is not defined

In [ ]:
import requests
import os
import pandas as pd

# List of team abbreviations
teams = [
    "ANA", "BOS", "BUF", "CAR", "CBJ", "CGY", "CHI", "COL", "DAL", "DET",
    "EDM", "FLA", "LAK", "MIN", "MTL", "NJD", "NSH", "NYI", "NYR", "OTT",
    "PHI", "PIT", "SEA", "SJS", "STL", "TBL", "TOR", "UTA", "VAN", "VGK",
    "WPG", "WSH"
]

def get_nhl_team_roster_and_save_csv(team):
    """
    Fetch NHL team roster and save it as a CSV file.

    Args:
        team (str): Team abbreviation (e.g., "ANA", "BOS").

    Saves:
        CSV file in NHL_data/rosters/<team>.csv containing the roster data.
    """
    # Define the URL and file path
    url = f"https://api-web.nhle.com/v1/roster/{team}/current"
    folder_path = "NHL_data/rosters"
    file_path = os.path.join(folder_path, f"{team}.csv")

    # Ensure the folder exists
    os.makedirs(folder_path, exist_ok=True)

    try:
        # Fetch the roster data
        resp = requests.get(
            url,
            timeout=30,
            allow_redirects=True,
            headers={"Accept": "application/json"},
        )
        resp.raise_for_status()
        roster_data = resp.json()
        # print(roster_data)
        # Extract player information
        players = roster_data.get("roster", [])
        if not players:
            print(f"No roster data found for team {team}.")
            return

        # Convert the player data to a DataFrame
        player_list = [
            {
                "Player Name": player.get("person", {}).get("fullName", "Unknown"),
                "Position": player.get("position", {}).get("abbreviation", "Unknown"),
                "Jersey Number": player.get("jerseyNumber", "Unknown"),
                "Team": team
            }
            for player in players
        ]
        df = pd.DataFrame(player_list)

        # Save the DataFrame as a CSV file
        df.to_csv(file_path, index=False)
        print(f"Roster for team {team} saved to {file_path}.")
    except requests.RequestException as e:
        print(f"Failed to fetch roster for team {team}: {e}")

# Fetch and save rosters for all teams
for team in teams:
    get_nhl_team_roster_and_save_csv(team)

ValueError: All arrays must be of the same length

In [13]:
import requests
import os
import pandas as pd

# List of team abbreviations
teams = [
    "ANA", "BOS", "BUF", "CAR", "CBJ", "CGY", "CHI", "COL", "DAL", "DET",
    "EDM", "FLA", "LAK", "MIN", "MTL", "NJD", "NSH", "NYI", "NYR", "OTT",
    "PHI", "PIT", "SEA", "SJS", "STL", "TBL", "TOR", "UTA", "VAN", "VGK",
    "WPG", "WSH"
]

def get_nhl_team_roster_and_save_csv(team):
    """
    Fetch NHL team roster and save it as a CSV file.

    Args:
        team (str): Team abbreviation (e.g., "ANA", "BOS").

    Saves:
        CSV file in NHL_data/rosters/<team>.csv containing the roster data.
    """
    # Define the URL and file path
    url = f"https://api-web.nhle.com/v1/roster/{team}/current"
    folder_path = "NHL_data/rosters"
    file_path = os.path.join(folder_path, f"{team}.csv")

    # Ensure the folder exists
    os.makedirs(folder_path, exist_ok=True)

    try:
        # Fetch the roster data
        resp = requests.get(
            url,
            timeout=30,
            allow_redirects=True,
            headers={"Accept": "application/json"},
        )
        resp.raise_for_status()
        roster_data = resp.json()

        # Combine players from all categories (forwards, defensemen, goalies)
        players = []
        for category in ["forwards", "defensemen", "goalies"]:
            category_players = roster_data.get(category, [])
            for player in category_players:
                players.append({
                    "Player Name": f"{player.get('firstName', {}).get('default', 'Unknown')} {player.get('lastName', {}).get('default', 'Unknown')}",
                    "Position": player.get("positionCode", "Unknown"),
                    "Jersey Number": player.get("sweaterNumber", "Unknown"),
                    "Height (in)": player.get("heightInInches", "Unknown"),
                    "Weight (lbs)": player.get("weightInPounds", "Unknown"),
                    "Birth Date": player.get("birthDate", "Unknown"),
                    "Birth City": player.get("birthCity", {}).get("default", "Unknown"),
                    "Birth Country": player.get("birthCountry", "Unknown"),
                    "Team": team,
                    "player_id": player.get("id","unknown")
                })

        # Check if there are players to save
        if not players:
            print(f"No roster data found for team {team}.")
            return

        # Convert the player data to a DataFrame
        df = pd.DataFrame(players)

        # Save the DataFrame as a CSV file
        df.to_csv(file_path, index=False)
        print(f"Roster for team {team} saved to {file_path}.")
    except requests.RequestException as e:
        print(f"Failed to fetch roster for team {team}: {e}")

# Fetch and save rosters for all teams
for team in teams:
    get_nhl_team_roster_and_save_csv(team)

Roster for team ANA saved to NHL_data/rosters/ANA.csv.
Roster for team BOS saved to NHL_data/rosters/BOS.csv.
Roster for team BUF saved to NHL_data/rosters/BUF.csv.
Roster for team CAR saved to NHL_data/rosters/CAR.csv.
Roster for team CBJ saved to NHL_data/rosters/CBJ.csv.
Roster for team CGY saved to NHL_data/rosters/CGY.csv.
Roster for team CHI saved to NHL_data/rosters/CHI.csv.
Roster for team COL saved to NHL_data/rosters/COL.csv.
Roster for team DAL saved to NHL_data/rosters/DAL.csv.
Roster for team DET saved to NHL_data/rosters/DET.csv.
Roster for team EDM saved to NHL_data/rosters/EDM.csv.
Roster for team FLA saved to NHL_data/rosters/FLA.csv.
Roster for team LAK saved to NHL_data/rosters/LAK.csv.
Roster for team MIN saved to NHL_data/rosters/MIN.csv.
Roster for team MTL saved to NHL_data/rosters/MTL.csv.
Roster for team NJD saved to NHL_data/rosters/NJD.csv.
Roster for team NSH saved to NHL_data/rosters/NSH.csv.
Roster for team NYI saved to NHL_data/rosters/NYI.csv.
Roster for

In [ ]:
# "past_games", 
# "past_sog", 
# "past_a_sog", 
# "past_e_shot", 
# "past_goals", 
# "past_a_goals", 
# "past_e_goals", 
# "past_on_ice_goal", 
# "past_a_on_ice_goal", 
# "past_assists1", 
# "past_assists2", 
# "past_rebound_goals"

# """
# - list of teams playing today
# - get roster of all teams
# - get players names for teams playing today
# - extract player data for each team playing today
# - go through each daily skaters file by newest first and compile a list accordingly
# """


['EDM', 'CGY', 'TOR', 'MTL', 'VGK', 'LAK', 'WSH', 'BOS']

In [ ]:
# ----------------------------------------------
# Get NHL roster by team
import importlib
import NHL_data_fetcher
importlib.reload(NHL_data_fetcher)

team = "CBJ"
nhl = NHL_data_fetcher.get_nhl_team_roster(team)

for player in nhl.get('forwards', []):
    first_name = player.get('firstName', {}).get('default')
    last_name = player.get('lastName', {}).get('default')
    print(first_name, last_name)

# --------------------------------------------
# get team by season
import importlib
import data_fetcher
importlib.reload(data_fetcher)

nhl = data_fetcher.get_nhl_team_by_season_type()

# --------------------------------------------
# get the schedule from api
import importlib
import NHL_data_fetcher
import data_processor
importlib.reload(NHL_data_fetcher)
importlib.reload(data_processor)

# nhl = NHL_data_fetcher.get_nhl_week_schedule_now()

# # convert schedule
# data_processor.export_schedule_to_csv(nhl,"NHL_data/nhl_schedule.csv")

nhl = NHL_data_fetcher.get_nhl_calendar_schedule_now()

# convert schedule
data_processor.export_schedule_to_csv(nhl,"NHL_data/nhl_calendar_schedule.csv")


# process schedule into seperate files
# NHL_script.process_full_schedule()
# ---------------------------------------
# STANDINGS
import importlib
import data_fetcher
importlib.reload(data_fetcher)

nhl = data_fetcher.get_nhl_standings_now()
# for x in nhl:
#     print(x, nhl[x])

for y in nhl['standings']:
    for z in y:
        print(z, y[z])
    print('=====================')

In [ ]:
#IGNORE FOR NOW

import pandas as pd

def generate_html_table():
    """
    Reads the NHL skaters CSV file and generates an HTML table.

    The HTML table is saved to the file path specified in `table_path`.

    File Paths:
    - Input: 'NHL_data/nhl_skaters_2024_2025_regular_latest.csv'
    - Output: 'NHL_data/skaters_table.html'
    """
    # File paths
    csv_path = "NHL_data/nhl_skaters_2024_2025_regular_latest.csv"
    table_path = "NHL_data/skaters_table.html"

    # Read the CSV file into a DataFrame
    df = pd.read_csv(csv_path)

    # Generate the HTML table
    html_table = df.to_html(index=False, classes="table table-striped", border=0)

    # Save the HTML table to the specified file
    with open(table_path, "w", encoding="utf-8") as file:
        file.write(html_table)

    print(f"HTML table saved to {table_path}")
generate_html_table()


HTML table saved to NHL_data/skaters_table.html
